# CLIMADA v6

## Download Hazard

In [ ]:
from climada_petals.hazard.rf_glofas import setup_all
from climada.util import log_level

with log_level("DEBUG"):
    setup_all()

In [ ]:
from climada_petals.hazard.rf_glofas import RiverFloodInundation, save_file
import xarray as xr

with xr.open_dataarray(
    "data/hazard/69dea9bf076a18556d1eee3641f185f9.grib", chunks="auto"
) as da:
    rf = RiverFloodInundation()
    # rf.download_forecast("Germany", "2021-06-13", 5)
    ds_flood = rf.compute(discharge=da)
    save_file(ds_flood, "data/hazard/flood-2021-06.nc", zlib=True, complevel=2)

In [1]:
import xarray as xr
from climada_petals.hazard.rf_glofas import hazard_series_from_dataset

with xr.open_dataset("data/hazard/flood-2021-06-fine.nc", chunks="auto") as ds:
    hazard = hazard_series_from_dataset(ds, intensity="flood_depth", event_dim="number")

2025-08-29 14:41:23,377 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:26,077 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:29,035 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:31,733 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:36,517 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:41,775 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:47,823 - climada.hazard.io - WARNING - Failed to read values of 'number' as dates. Hazard.event_name will be empty strings
2025-08-29 14:41:49,

In [6]:
from climada.entity.exposures import Exposures
from climada.entity.impact_funcs import ImpactFunc, ImpactFuncSet

exposures = [
    Exposures.from_raster("data/exposure/deu_ppp_2000_1km_Aggregated_UNadj.tif"),
    Exposures.from_raster("data/exposure/deu_ppp_2001_1km_Aggregated_UNadj.tif"),
]
for exp in exposures:
    exp.gdf["impf_RF"] = 1

exposures[0].assign_centroids(hazard.iloc[0])
exposures[1].gdf["centr_RF"] = exposures[0].gdf["centr_RF"]

impf = ImpactFuncSet(
    [
        ImpactFunc.from_poly_s_shape(
            haz_type="RF",
            intensity=(0, 10, 100),
            threshold=0.1,
            half_point=1.0,
            scale=1.0,
            exponent=3,
        )
    ]
)

In [3]:
# import xarray as xr
# import geopandas as gpd

# with xr.open_dataarray(
#     "data/exposure/deu_ppp_2000_UNadj.tif",
#     # "data/exposure/deu_ppp_2000_1km_Aggregated_UNadj.tif",
#     chunks="auto",
#     decode_coords="all",
# ) as da:
#     # da = da.assign_coords(x=lambda x: x.astype("float32"), y=lambda y: y.astype("float32"))
#     da = da.squeeze(drop=True).stack(lon_lat=["x", "y"], create_index=False)
#     da = da.compute()
#     da = da[da > 0]

#     gdf = gpd.GeoDataFrame(
#         data={"value": da}, geometry=gpd.points_from_xy(da["x"], da["y"])
#     )
# del da
# gdf

In [4]:
# from climada.entity.exposures import Exposures

# exposure = Exposures(data=gdf, copy=False)
# exposure.gdf["impf_RF"] = 1

In [7]:
from climada.engine import ImpactCalc

for exp in exposures:
    for haz in hazard:
        impact = ImpactCalc(exp, impf, haz).impact(
            save_mat=False, assign_centroids=False
        )
    # impact.write_hdf5("data/output/impact-oldstyle.h5")